In [37]:
import pandas as pd

df = pd.read_csv("../../data/processed/hotel_general_info_fixed.csv")

In [38]:
# %pip install geopy

In [39]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import pandas as pd

geolocator = Nominatim(
    user_agent="hotel_geocoder_vn",
    timeout=10   # ⬅️ tăng từ mặc định (1s) lên 10s
)
geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1.5,
    max_retries=2,
    error_wait_seconds=5,
    swallow_exceptions=True
)

def geocode_address(address):
    try:
        location = geocode(address, language="vi")
        if location:
            return location.latitude, location.longitude
    except Exception:
        pass
    return None, None



def shorten_address(addr):
    if pd.isna(addr) or addr.strip() == "":
        return None

    parts = [p.strip() for p in addr.split(",")]

    keep = []
    for p in parts:
        pl = p.lower()

        if any(k in pl for k in ["đường", "đại lộ", "street"]):
            keep.append(p)
        elif any(k in pl for k in ["phường", "quận", "thành phố", "tỉnh"]):
            keep.append(p)
        elif "vietnam" in pl:
            keep.append("Vietnam")

    return ", ".join(dict.fromkeys(keep))



In [40]:
#df = df[:20].copy()
df["geopy_address"] = df["hotel_address"].apply(shorten_address)
df[["lat", "lng"]] = df["geopy_address"].apply(
    lambda x: pd.Series(geocode_address(x))
)

In [41]:
total_rows = len(df)
print("Tổng số dòng:", total_rows)

success_count = df["lat"].notna() & df["lng"].notna()
success_count = success_count.sum()
print("Số dòng có lat/lng:", success_count)
fail_count = total_rows - success_count
print("Số dòng bị NaN:", fail_count)
print(f"Tỉ lệ thành công: {success_count / total_rows:.2%}")
print(f"Tỉ lệ thất bại: {fail_count / total_rows:.2%}")

Tổng số dòng: 5207
Số dòng có lat/lng: 2068
Số dòng bị NaN: 3139
Tỉ lệ thành công: 39.72%
Tỉ lệ thất bại: 60.28%


In [42]:
df.columns

Index(['hotel_id', 'hotel_name', 'hotel_address', 'region', 'geopy_address',
       'lat', 'lng'],
      dtype='object')

In [43]:
df[['hotel_address','lat', 'lng']]

,hotel_address,lat,lng
0,"73 Đoàn Thị Điểm Bà Rịa, Việt Nam",NaN,NaN
1,"Ba Ria City, Long Huong Phường, Việt Nam",10.505619,107.130630
2,"QL51, Việt Nam",NaN,NaN
3,"328C, Phường Hưng Định, Thành phố Thuận An, Bì...",10.940005,106.691635
4,"153 Hoang Van Thu Đường, Bình Dương, Việt Nam",22.826861,104.988444
...,...,...,...
5202,"273 Lê Văn Phẩm Mỹ Tho Tiền Giang, Tiền Giang,...",NaN,NaN
5203,"Mỹ Tho, Tiền Giang, Việt Nam",NaN,NaN
5204,"Mỹ Tho, Tiền Giang, Việt Nam",NaN,NaN
5205,"ngõ 63 Đường Phạm Hùng, Việt Nam",21.014173,105.784808


In [44]:
# df.to_csv("hotel_general_info_updated_geopy.csv", index=False, encoding="utf-8-sig")